In [29]:
# Getting Data from Github
!wget -q https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/2-adverse-events.csv

# Verify the files are downloaded
!ls -lh 2-adverse-events.csv

-rw-r--r-- 1 root root 588K Feb 16 21:19 2-adverse-events.csv


In [30]:
# Import the needed tools to configure and control the system from pyspark library
from pyspark import SparkConf, SparkContext

# Create a 'Configuration' object to define how the program should run
# .setMaster("local") is commanding the code to run on your own computer
# .setAppName("Adverse_Event") gives the specific task a name for the logs
conf = SparkConf().setMaster("local").setAppName("Adverse_Arm")

# Creating the 'Spark Context', which is an entry point for all Spark functions
# It takes the configuration, and starts the engine
sc = SparkContext(conf = conf)

In [31]:
def normalize(name):
  name = name.replace("_", " ") # replace underscores with spaces
  name = name.strip().upper() # remove spaces and make it to uppercase
  name = " ".join(name.split()) # collapse multiple spaces
  return name

# Define a function to parse each line of the CSV file
def parseLine(filterLine):
  # Split the line by commas
  fields = filterLine.split(',')
  # Extract the adverse_event and arm_affected
  adverseEvent = normalize(fields[1])
  armAffected= int(fields[2])
  armAtRisk = int(fields[3])
  # Return a tuple (adverse_event, arm_affected)
  return (adverseEvent, armAffected, armAtRisk)

# Load the CSV file as an RDD
lines = sc.textFile("2-adverse-events.csv")

# Remove the header line
filterLine = lines.filter(lambda x: "allergic_ID" not in x)

# parse each line
rdd = filterLine.map(parseLine)

# summarize affected patients vs at_risk patients
TotAffected = rdd.map(lambda x: (x[0], (float(x[1]), 1))).reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
print(TotAffected.first())

TotAtRisk = rdd.map(lambda x: (x[0], (float(x[1]), float(x[2])))).reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
print(TotAtRisk.first())

print('\n')

# avg. affected arm count per arm
avgAffected = TotAffected.mapValues(lambda x: (x[0]/x[1]))
avgresults = avgAffected.collect()
for result in avgresults:
  # print("Avg. affected count per arm: " + result[0])
  print("Avg. affected count per arm: " + result[0] + " " + '{:.2%}'.format(result[1]))

print('\n')

# affected rate (affected/at_risk)
affctedRate = TotAtRisk.mapValues(lambda x: (x[0]/x[1]))
results = affctedRate.collect()
for result in results:
  print("Affected Rate: " + result[0] + " " +'{:.2%}'.format(result[1]))

# Stop the SparkContext
sc.stop()

('DRUG HYPERSENSITIVITY', (1893.0, 499))
('DRUG HYPERSENSITIVITY', (1893.0, 116368.0))


Avg. affected count per arm: DRUG HYPERSENSITIVITY 379.36%
Avg. affected count per arm: ANAPHYLAXIS 980.18%
Avg. affected count per arm: ALLERGIC REACTION 580.31%
Avg. affected count per arm: ALLERGY 345.45%
Avg. affected count per arm: DERMATITIS ALLERGIC 136.97%
Avg. affected count per arm: HYPERSENSITIVITY 326.37%


Affected Rate: DRUG HYPERSENSITIVITY 1.63%
Affected Rate: ANAPHYLAXIS 4.49%
Affected Rate: ALLERGIC REACTION 3.61%
Affected Rate: ALLERGY 1.41%
Affected Rate: DERMATITIS ALLERGIC 0.51%
Affected Rate: HYPERSENSITIVITY 1.76%
